In [ ]:
import fitz
import pytesseract
from PIL import Image
import io
import cv2
import numpy as np
import re
import matplotlib.pyplot as plt
import os

class PDFProcessor:
    def __init__(self):
        self.current_page = None
        self.current_image = None
        self.crop_coords = {}
        self.output_dir = "processed_pagesss"
        
    def show_page_with_grid(self, image, page_num):
        """Display a page with grid overlay and coordinate tracking"""
        self.current_page = page_num
        self.current_image = image
        
        width, height = image.size
        
        while True:
            # Create figure and plot image
            fig, ax = plt.subplots(figsize=(15, 10))
            ax.imshow(image)
            
            # Add grid
            ax.grid(True, color='red', alpha=0.3)
            
            # Add major grid lines every 100 pixels
            major_x_ticks = np.arange(0, width, 100)
            major_y_ticks = np.arange(0, height, 100)
            ax.set_xticks(major_x_ticks)
            ax.set_yticks(major_y_ticks)
            
            # Add labels
            plt.title(f"Page {page_num + 1} - Move mouse to see coordinates\nDimensions: {width}x{height}")
            
            # Add coordinate display
            coord_text = ax.text(0.02, 0.98, '', transform=ax.transAxes, 
                               bbox=dict(facecolor='white', alpha=0.8),
                               verticalalignment='top')
            
            def mouse_move(event):
                if event.inaxes:
                    x, y = int(event.xdata), int(event.ydata)
                    coord_text.set_text(f'Coordinates: x={x}, y={y}')
                    plt.draw()
            
            fig.canvas.mpl_connect('motion_notify_event', mouse_move)
            
            plt.show()
            
            print("\nOptions:")
            print("1. Enter crop coordinates")
            print("2. Skip this page")
            print("3. Show grid view again")
            choice = input("Enter your choice (1, 2, or 3): ")
            
            if choice == "2":
                self.crop_coords[self.current_page] = None
                print(f"Skipping page {page_num + 1}")
                break
            elif choice == "3":
                continue
            elif choice == "1":
                try:
                    print("\nEnter coordinates for cropping:")
                    print("Format: x1,y1,x2,y2 (top-left and bottom-right corners)")
                    coords_input = input("Coordinates: ")
                    x1, y1, x2, y2 = map(int, coords_input.split(','))
                    
                    # Ensure coordinates are within image bounds
                    x1 = max(0, min(x1, width))
                    x2 = max(0, min(x2, width))
                    y1 = max(0, min(y1, height))
                    y2 = max(0, min(y2, height))
                    
                    # Show crop preview
                    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
                    
                    # Show full image with crop rectangle
                    ax1.imshow(image)
                    rect = plt.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, color='red')
                    ax1.add_patch(rect)
                    ax1.set_title("Full Page with Crop Area")
                    
                    # Show cropped preview
                    cropped_preview = image.crop((x1, y1, x2, y2))
                    ax2.imshow(cropped_preview)
                    ax2.set_title("Cropped Preview")
                    
                    plt.show()
                    
                    print("\nIs this crop okay?")
                    print("1. Accept crop")
                    print("2. Try again")
                    confirm = input("Enter your choice (1 or 2): ")
                    
                    if confirm == "1":
                        self.crop_coords[self.current_page] = (x1, y1, x2, y2)
                        break
                    
                except ValueError:
                    print("Invalid coordinates format. Please try again.")
                    continue
                
    def process_pdf(self, pdf_path):
        """Process PDF with user interaction for cropping"""
        if not os.path.exists(self.output_dir):
            os.makedirs(self.output_dir)
            
        doc = fitz.open(pdf_path)
        all_items = []
        
        for page_num in range(len(doc)):
            page = doc[page_num]
            
            # Increase the resolution of the rendered page
            zoom = 4  # Increased zoom factor for better resolution
            mat = fitz.Matrix(zoom, zoom)
            pix = page.get_pixmap(matrix=mat, alpha=False)
            
            # Convert PyMuPDF pixmap to PIL Image
            img_data = pix.samples
            img = Image.frombytes("RGB", [pix.width, pix.height], img_data)
            
            # Show page and get user crop selection
            print(f"\nProcessing page {page_num + 1}")
            self.show_page_with_grid(img, page_num)
            
            if self.crop_coords.get(page_num) is None:
                print(f"Skipping page {page_num + 1}")
                continue
                
            # Scale crop coordinates according to zoom factor
            x1, y1, x2, y2 = [int(coord) for coord in self.crop_coords[page_num]]
            cropped_img = img.crop((x1, y1, x2, y2))
            
            # Save cropped image with high DPI
            output_path = os.path.join(self.output_dir, f'page_{page_num + 1}.png')
            cropped_img.save(output_path, dpi=(300, 300))
            print(f"Saved cropped image to: {output_path}")
            
            # Enhanced preprocessing for better OCR
            processed_img = self.preprocess_image(cropped_img)
            
            # Save processed image for verification
            processed_path = os.path.join(self.output_dir, f'page_{page_num + 1}_processed.png')
            processed_img.save(processed_path, dpi=(300, 300))
            
            # Perform OCR with improved settings
            custom_config = r'--oem 3 --psm 6 -c tessedit_char_whitelist=0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ/\"\\-. '
            text = pytesseract.image_to_string(processed_img, config=custom_config)
            
            # Extract and store BOM data
            items = self.extract_bom_data(text)
            all_items.extend(items)
        
        doc.close()
        return all_items
    
    @staticmethod
    def preprocess_image(image):
        """Enhanced preprocessing for better text clarity"""
        # Convert to OpenCV format
        opencv_img = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)
        
        # Convert to grayscale
        gray = cv2.cvtColor(opencv_img, cv2.COLOR_BGR2GRAY)
        
        # Apply adaptive thresholding
        binary = cv2.adaptiveThreshold(
            gray,
            255,
            cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv2.THRESH_BINARY,
            11,  # Block size
            2    # C constant
        )
        
        # Denoise
        denoised = cv2.fastNlMeansDenoising(binary)
        
        # Optional: Apply slight sharpening
        kernel = np.array([[-1,-1,-1],
                         [-1, 9,-1],
                         [-1,-1,-1]])
        sharpened = cv2.filter2D(denoised, -1, kernel)
        
        return Image.fromarray(sharpened)
    
    @staticmethod
    def extract_bom_data(text):
        """Extract Bill of Materials data"""
        items = []
        lines = [line.strip() for line in text.split('\n') if line.strip()]
        
        pattern = r'(?P<id>\d+)\s*(?P<qty>[\d\'-]+)\s*(?P<location>SHOP|FIELD)\s*(?P<nd>[\d/]+\"?)\s*(?P<description>.*)'
        
        for line in lines:
            match = re.search(pattern, line)
            if match:
                items.append({
                    'id': match.group('id'),
                    'qty': match.group('qty'),
                    'location': match.group('location'),
                    'nd': match.group('nd'),
                    'description': match.group('description').strip()
                })
        
        return items

def display_results(items):
    """Display the BOM results in a table format"""
    if not items:
        print("\nNo items were extracted from the PDF.")
        return
        
    print("\nBill of Materials:")
    print("=" * 100)
    print(f"{'ID':<5}{'QTY':<10}{'SHOP/FIELD':<12}{'ND':<8}{'DESCRIPTION':<65}")
    print("-" * 100)
    
    for item in items:
        print(f"{item['id']:<5}{item['qty']:<10}{item['location']:<12}{item['nd']:<8}{item['description']:<65}")

def main():
    print("PDF Bill of Materials Processor")
    print("==============================")
    pdf_path = input("Enter the path to your PDF file: ")
    
    if not os.path.exists(pdf_path):
        print(f"Error: File not found at {pdf_path}")
        return
        
    try:
        processor = PDFProcessor()
        items = processor.process_pdf(pdf_path)
        display_results(items)
        
        print(f"\nProcessed images have been saved to the '{processor.output_dir}' directory")
        print("Both original crops and processed versions are saved for comparison.")
        
    except Exception as e:
        print(f"Error processing PDF: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

In [5]:
import fitz
import pytesseract
from PIL import Image
import cv2
import numpy as np
import re
import os

class PDFProcessor:
    def __init__(self):
        self.output_dir = "zoom"
        
    def find_table_region(self, img_array):
        """Find the region containing 'DESCRIPTION' and return coordinates for the table area"""
        # Convert to grayscale if not already
        if len(img_array.shape) == 3:
            gray = cv2.cvtColor(img_array, cv2.COLOR_BGR2GRAY)
        else:
            gray = img_array
            
        # Run OCR to find "DESCRIPTION"
        text_data = pytesseract.image_to_data(gray, output_type=pytesseract.Output.DICT)
        
        # Find "DESCRIPTION" coordinates
        for i, text in enumerate(text_data['text']):
            if 'DESCRIPTION' in text:
                x = text_data['left'][i]
                y = text_data['top'][i]
                # Expand region to capture full table
                # Adjust these values based on your typical table size
                table_width = img_array.shape[1]  # Full width
                table_height = int(img_array.shape[0] * 0.2)  # 20% of height
                return (0, max(0, y - 50), table_width, y + table_height)
                
        # If not found, return None
        return None

    def process_pdf(self, pdf_path):
        """Process PDF automatically without user interaction"""
        if not os.path.exists(self.output_dir):
            os.makedirs(self.output_dir)
            
        doc = fitz.open(pdf_path)
        all_items = []
        
        for page_num in range(len(doc)):
            page = doc[page_num]
            
            # Increase the resolution of the rendered page
            zoom = 4  # Increased zoom factor for better resolution
            mat = fitz.Matrix(zoom, zoom)
            pix = page.get_pixmap(matrix=mat, alpha=False)
            
            # Convert PyMuPDF pixmap to PIL Image
            img_data = pix.samples
            img = Image.frombytes("RGB", [pix.width, pix.height], img_data)
            
            print(f"\nProcessing page {page_num + 1}")
            
            # Convert PIL image to OpenCV format for processing
            opencv_img = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
            
            # Find the table region
            table_coords = self.find_table_region(opencv_img)
            
            if table_coords:
                x, y, w, h = table_coords
                # Crop the image to the table region for saving
                table_region = opencv_img[y:y+h, x:w]
                
                # Process the cropped region
                processed_img = self.preprocess_image(Image.fromarray(cv2.cvtColor(table_region, cv2.COLOR_BGR2RGB)))
            else:
                # If table region not found, process the whole page
                processed_img = self.preprocess_image(img)
            
            # Save only the processed image
            processed_path = os.path.join(self.output_dir, f'page_{page_num + 1}.png')
            processed_img.save(processed_path, dpi=(300, 300))
            
            # Perform OCR on the full page image for data extraction
            custom_config = r'--oem 3 --psm 6 -c tessedit_char_whitelist=0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ/\"\\-. '
            text = pytesseract.image_to_string(img, config=custom_config)
            
            # Extract and store BOM data
            items = self.extract_bom_data(text)
            all_items.extend(items)
        
        doc.close()
        return all_items
    
    @staticmethod
    def preprocess_image(image):
        """Simple preprocessing for better text clarity"""
        # Convert to OpenCV format
        opencv_img = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)
        
        # Convert to grayscale
        gray = cv2.cvtColor(opencv_img, cv2.COLOR_BGR2GRAY)
        
        # Simple binary threshold
        _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        
        return Image.fromarray(binary)
    
    @staticmethod
    def extract_bom_data(text):
        """Extract Bill of Materials data"""
        items = []
        lines = [line.strip() for line in text.split('\n') if line.strip()]
        
        pattern = r'(?P<id>\d+)\s*(?P<qty>[\d\'-]+)\s*(?P<location>SHOP|FIELD)\s*(?P<nd>[\d/]+\"?)\s*(?P<description>.*)'
        
        for line in lines:
            match = re.search(pattern, line)
            if match:
                items.append({
                    'id': match.group('id'),
                    'qty': match.group('qty'),
                    'location': match.group('location'),
                    'nd': match.group('nd'),
                    'description': match.group('description').strip()
                })
        
        return items

def display_results(items):
    """Display the BOM results in a table format"""
    if not items:
        print("\nNo items were extracted from the PDF.")
        return
        
    print("\nBill of Materials:")
    print("=" * 100)
    print(f"{'ID':<5}{'QTY':<10}{'SHOP/FIELD':<12}{'ND':<8}{'DESCRIPTION':<65}")
    print("-" * 100)
    
    for item in items:
        print(f"{item['id']:<5}{item['qty']:<10}{item['location']:<12}{item['nd']:<8}{item['description']:<65}")

def main():
    print("PDF Bill of Materials Processor")
    print("==============================")
    pdf_path = input("Enter the path to your PDF file: ")
    
    if not os.path.exists(pdf_path):
        print(f"Error: File not found at {pdf_path}")
        return
        
    try:
        processor = PDFProcessor()
        items = processor.process_pdf(pdf_path)
        display_results(items)
        
        print(f"\nProcessed images have been saved to the '{processor.output_dir}' directory")
        
    except Exception as e:
        print(f"Error processing PDF: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

PDF Bill of Materials Processor

Processing page 1

Processing page 2

Processing page 3

Processing page 4

Processing page 5

Processing page 6

Processing page 7

Processing page 8

Processing page 9

Processing page 10

Processing page 11

Processing page 12

Processing page 13

Processing page 14

Processing page 15


KeyboardInterrupt: 